In [0]:
# Databricks notebook source

# ==========================================================
# Bronze Data Quality Checks
#
# Notebook:
# 01_bronze_dq_checks
#
# Purpose:
# Validate Bronze ingestion before Silver processing.
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC

CATALOG = "crypto_pipeline"

BRONZE_TABLE = f"{CATALOG}.bronze.raw_coin_market_data"
DQ_TABLE = f"{CATALOG}.meta.dq_results"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create DQ Results Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {DQ_TABLE}

(

layer STRING,

check_name STRING,

status STRING,

failed_rows INT,

check_timestamp TIMESTAMP

)

USING DELTA

""")

bronze_df = spark.table(BRONZE_TABLE)

dq_logs = []

# ==========================================================
# Helper
# ==========================================================

def log_result(layer, check_name, failed_rows):

    status = "PASS" if failed_rows == 0 else "FAIL"

    dq_logs.append(

        Row(

            layer=layer,

            check_name=check_name,

            status=status,

            failed_rows=int(failed_rows),

            check_timestamp=datetime.now(UTC)

        )

    )

In [0]:
duplicates = (

    bronze_df

    .groupBy(

        "id",

        "_batch_id"

    )

    .count()

    .filter("count > 1")

)

dup_count = duplicates.count()

log_result(

    "bronze",

    "Duplicate Coin IDs",

    dup_count

)

In [0]:
null_id = (

    bronze_df

    .filter(

        F.col("id").isNull()

    )

)

null_count = null_id.count()

log_result(

    "bronze",

    "Null Coin ID",

    null_count

)

In [0]:
null_price = (

    bronze_df

    .filter(

        F.col("current_price").isNull()

    )

)

null_price_count = null_price.count()

log_result(

    "bronze",

    "Null Current Price",

    null_price_count

)

In [0]:
negative_price = (

    bronze_df

    .filter(

        F.col("current_price") < 0

    )

)

negative_count = negative_price.count()

log_result(

    "bronze",

    "Negative Price",

    negative_count

)

In [0]:
null_ts = (

    bronze_df

    .filter(

        F.col("_ingested_at").isNull()

    )

)

null_ts_count = null_ts.count()

log_result(

    "bronze",

    "Null Ingestion Timestamp",

    null_ts_count

)

In [0]:
null_batch = (

    bronze_df

    .filter(

        F.col("_batch_id").isNull()

    )

)

null_batch_count = null_batch.count()

log_result(

    "bronze",

    "Null Batch ID",

    null_batch_count

)

In [0]:
batch_sizes = (

    bronze_df

    .groupBy("_batch_id")

    .count()

)

empty_batches = batch_sizes.filter("count = 0")

empty_batch_count = empty_batches.count()

log_result(

    "bronze",

    "Empty Batch",

    empty_batch_count

)

In [0]:
spark.createDataFrame(dq_logs).write.mode("append").saveAsTable(DQ_TABLE)

display(

    spark.table(DQ_TABLE)

    .orderBy(

        F.desc("check_timestamp")

    )

)

layer,check_name,status,failed_rows,check_timestamp
bronze,Empty Batch,PASS,0,2026-07-22T11:58:08.574Z
bronze,Null Batch ID,PASS,0,2026-07-22T11:58:03.279Z
bronze,Null Ingestion Timestamp,PASS,0,2026-07-22T11:57:58.212Z
bronze,Negative Price,PASS,0,2026-07-22T11:57:52.372Z
bronze,Null Current Price,PASS,0,2026-07-22T11:57:47.503Z
bronze,Null Coin ID,PASS,0,2026-07-22T11:57:42.010Z
bronze,Duplicate Coin IDs,PASS,0,2026-07-22T11:57:35.768Z
gold,Invalid Movement,PASS,0,2026-07-22T11:55:40.259Z
gold,Invalid Rank,PASS,0,2026-07-22T11:55:33.695Z
gold,Null Rolling Average,PASS,0,2026-07-22T11:55:27.875Z


In [0]:
failed = sum(

    row.failed_rows

    for row in dq_logs

)

if failed > 0:

    raise Exception(

        f"Bronze DQ Failed. Failed Rows = {failed}"

    )

print("Bronze Data Quality Passed Successfully.")

Bronze Data Quality Passed Successfully.
